# Xử lý dữ liệu

In [53]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"F:\project\NCKH\Data\tiki_reviews_with_sentiment_final.csv")

# Read CSV (try UTF-8 first; fall back to UTF-8 with BOM)
try:
    df = pd.read_csv(csv_path, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(csv_path, encoding="utf-8-sig")

# Drop columns: any containing '_score_' and specific columns
score_cols = [col for col in df.columns if "_score_" in col]
explicit_drop = ["content", "customer_name", "created_at", "review_id", "segmented_content"]
cols_to_drop = score_cols + explicit_drop
existing_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing_to_drop)

print("Dropped columns (existing):")
print(sorted(existing_to_drop))

print("\nFirst 10 rows (after dropping columns):")
display(df.head(10))

print("\nRemaining columns:")
print(list(df.columns))

Dropped columns (existing):
['as_content_score_na', 'as_content_score_negative', 'as_content_score_neutral', 'as_content_score_positive', 'as_delivery_score_na', 'as_delivery_score_negative', 'as_delivery_score_neutral', 'as_delivery_score_positive', 'as_packaging_score_na', 'as_packaging_score_negative', 'as_packaging_score_neutral', 'as_packaging_score_positive', 'as_physical_score_na', 'as_physical_score_negative', 'as_physical_score_neutral', 'as_physical_score_positive', 'as_price_score_na', 'as_price_score_negative', 'as_price_score_neutral', 'as_price_score_positive', 'as_service_score_na', 'as_service_score_negative', 'as_service_score_neutral', 'as_service_score_positive', 'content', 'created_at', 'customer_name', 'review_id', 'segmented_content']

First 10 rows (after dropping columns):


,product_id,rating,likes,as_content_label_id,as_physical_label_id,as_price_label_id,as_packaging_label_id,as_delivery_label_id,as_service_label_id
0,270880693,5,7,3,3,3,3,3,3
1,270880693,5,0,3,2,3,2,2,3
2,270880693,5,3,2,3,3,3,3,3
3,270880693,5,3,2,3,3,3,3,3
4,270880693,5,0,2,3,3,3,3,3
5,270880693,5,5,2,3,3,3,3,3
6,270880693,5,3,2,3,3,3,3,3
7,270880693,5,3,2,3,3,3,3,3
8,270880693,5,5,2,3,3,3,3,3
9,270880693,5,3,2,3,3,3,3,3



Remaining columns:
['product_id', 'rating', 'likes', 'as_content_label_id', 'as_physical_label_id', 'as_price_label_id', 'as_packaging_label_id', 'as_delivery_label_id', 'as_service_label_id']


In [54]:
# Aggregate label distributions per product_id (+ average rating)
import re

if "df" not in globals():
    raise NameError("df is not defined. Run the data-loading cell first.")

required_cols = {"product_id", "rating"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns in df: {sorted(missing)}")

label_cols = [c for c in df.columns if "_label_" in c]
print("Label columns:", label_cols)

def _base_label_name(col_name: str) -> str:
    # Example: 'as_content_label_id' -> 'content_label'
    name = re.sub(r"^as_", "", col_name)
    name = re.sub(r"_id$", "", name)
    name = re.sub(r"_label$", "_label", name)
    name = re.sub(r"_label_id$", "_label", name)
    return name

# Number of reviews per product_id (duplicates == multiple reviews)
df_by_product = df.groupby("product_id").size().rename("review_count").to_frame()

# Average rating per product_id
rating_numeric = pd.to_numeric(df["rating"], errors="coerce")
avg_rating = rating_numeric.groupby(df["product_id"]).mean().rename("avg_rating")
df_by_product = df_by_product.join(avg_rating, how="left")

# For each *_label_* column, create count columns like content_label_0, content_label_1, ...
for col in label_cols:
    base = _base_label_name(col)
    tmp = df[["product_id", col]].copy()
    tmp[col] = tmp[col].fillna("NA")
    counts = tmp.groupby(["product_id", col]).size().unstack(fill_value=0)
    counts = counts.rename(columns={val: f"{base}_{val}" for val in counts.columns})
    df_by_product = df_by_product.join(counts, how="left")

# Fill missing counts with 0 (keep avg_rating as NaN if it cannot be computed)
non_rating_cols = [c for c in df_by_product.columns if c != "avg_rating"]
df_by_product[non_rating_cols] = df_by_product[non_rating_cols].fillna(0)

print("\nAggregated shape:", df_by_product.shape)
display(df_by_product.head())

Label columns: ['as_content_label_id', 'as_physical_label_id', 'as_price_label_id', 'as_packaging_label_id', 'as_delivery_label_id', 'as_service_label_id']

Aggregated shape: (1435, 26)


,review_count,avg_rating,content_label_0,content_label_1,content_label_2,content_label_3,...,delivery_label_2,delivery_label_3,service_label_0,service_label_1,service_label_2,service_label_3
product_id,,,,,,,,,,,,,
335337,194,4.592784,3,4,82,105,...,44,143,5,0,10,179
347334,202,4.673267,1,5,83,113,...,47,147,1,0,10,191
356317,8,4.250000,0,0,6,2,...,0,8,0,0,0,8
377644,181,4.685083,5,12,105,59,...,34,145,1,0,6,174
381533,12,4.750000,0,0,8,4,...,0,12,0,0,0,12


In [55]:
# Export aggregated result to Excel (.xlsx)
from pathlib import Path
from datetime import datetime

if "df_by_product" not in globals():
    raise NameError("df_by_product is not defined. Run Cell 3 first.")

out_path = Path(r"F:\project\NCKH\Data\exports\df_by_product.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_export = df_by_product.reset_index()  # bring product_id back as a column

try:
    df_export.to_excel(out_path, index=False, engine="openpyxl")
    saved_path = out_path
except PermissionError:
    # Usually happens when the target .xlsx is open in Excel (file lock).
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    saved_path = out_path.with_name(f"{out_path.stem}_alt_{ts}{out_path.suffix}")
    df_export.to_excel(saved_path, index=False, engine="openpyxl")

print(f"Saved: {saved_path}")
print(f"Rows: {len(df_export):,} | Columns: {df_export.shape[1]}")

Saved: F:\project\NCKH\Data\exports\df_by_product_alt_20260407_131915.xlsx
Rows: 1,435 | Columns: 27


In [59]:
# Join aggregated dataframe with product metadata (tiki_products.csv)
import pandas as pd
from pathlib import Path

if "df_by_product" not in globals():
    raise NameError("df_by_product is not defined. Run Cell 3 first.")

products_path = Path(r"F:\project\NCKH\Data\tiki_products.csv")

# Read products CSV with a couple of common encodings
try:
    products = pd.read_csv(products_path, encoding="utf-8")
except UnicodeDecodeError:
    products = pd.read_csv(products_path, encoding="utf-8-sig")

# Resolve ID column name in products file
if "ID" in products.columns:
    products_id_col = "ID"
elif "id" in products.columns:
    products_id_col = "id"
else:
    raise KeyError("Cannot find 'ID' (or 'id') column in tiki_products.csv")

# Prepare keys (coerce to numeric for a robust join)
agg = df_by_product.reset_index()  # product_id becomes a column
agg_key = pd.to_numeric(agg["product_id"], errors="coerce")
prod_key = pd.to_numeric(products[products_id_col], errors="coerce")

agg = agg.assign(_join_id=agg_key)
products = products.assign(_join_id=prod_key)

df_joined = agg.merge(products, on="_join_id", how="left", suffixes=("", "_prod"))
df_joined = df_joined.drop(columns=["_join_id"], errors="ignore")

# Drop unwanted columns after join
cols_drop_after_join = [
    "Number Reviews",
    "Rating Average",
    "URL",
    "ID",
    "Price (Before Discount)",
    "Price (After Discount)",
    "Name",
    "Seller ID",
    "id",  # in case the file uses lowercase
 ]
df_joined = df_joined.drop(columns=cols_drop_after_join, errors="ignore")

print("Joined shape:", df_joined.shape)
print("\nColumns after join (and drop):")
print(list(df_joined.columns))

display(df_joined.head())

Joined shape: (1435, 29)

Columns after join (and drop):
['product_id', 'review_count', 'avg_rating', 'content_label_0', 'content_label_1', 'content_label_2', 'content_label_3', 'physical_label_0', 'physical_label_1', 'physical_label_2', 'physical_label_3', 'price_label_0', 'price_label_1', 'price_label_2', 'price_label_3', 'packaging_label_0', 'packaging_label_1', 'packaging_label_2', 'packaging_label_3', 'delivery_label_0', 'delivery_label_1', 'delivery_label_2', 'delivery_label_3', 'service_label_0', 'service_label_1', 'service_label_2', 'service_label_3', 'Sales Count', 'Discount Rate']


,product_id,review_count,avg_rating,content_label_0,content_label_1,content_label_2,...,service_label_0,service_label_1,service_label_2,service_label_3,Sales Count,Discount Rate
0,335337,194,4.592784,3,4,82,...,5,0,10,179,1647,0
1,347334,202,4.673267,1,5,83,...,1,0,10,191,24533,26
2,356317,8,4.250000,0,0,6,...,0,0,0,8,415,25
3,377644,181,4.685083,5,12,105,...,1,0,6,174,38135,30
4,381533,12,4.750000,0,0,8,...,0,0,0,12,6,0


In [60]:
# Export joined result to Excel (.xlsx)
from pathlib import Path
from datetime import datetime

if "df_joined" not in globals():
    raise NameError("df_joined is not defined. Run the join cell first.")

out_path_joined = Path(r"F:\project\NCKH\Data\exports\df_joined.xlsx")
out_path_joined.parent.mkdir(parents=True, exist_ok=True)

try:
    df_joined.to_excel(out_path_joined, index=False, engine="openpyxl")
    saved_path = out_path_joined
except PermissionError:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    saved_path = out_path_joined.with_name(f"{out_path_joined.stem}_alt_{ts}{out_path_joined.suffix}")
    df_joined.to_excel(saved_path, index=False, engine="openpyxl")

print(f"Saved: {saved_path}")
print(f"Rows: {len(df_joined):,} | Columns: {df_joined.shape[1]}")

Saved: F:\project\NCKH\Data\exports\df_joined.xlsx
Rows: 1,435 | Columns: 29


# Phân tích tương quan đối với các nhãn tích tiêu cực

In [58]:
import pandas as pd

path = r"F:\project\NCKH\Data\exports\df_joined.xlsx"
df_joined_from_xlsx = pd.read_excel(path)

# Drop all columns ending with '_label_1' (only for this view)
label_1_cols = [c for c in df_joined_from_xlsx.columns if c.endswith("_label_1")]
df_joined_from_xlsx = df_joined_from_xlsx.drop(columns=label_1_cols, errors="ignore")

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 30)
pd.set_option("display.width", 200)

print(df_joined_from_xlsx.head(10))

   product_id  review_count  avg_rating  content_label_0  content_label_2  content_label_3  ...  delivery_label_3  service_label_0  service_label_2  service_label_3  Sales Count  Discount Rate
0      335337           194    4.592784                3               82              105  ...               143                5               10              179         1647              0
1      347334           202    4.673267                1               83              113  ...               147                1               10              191        24533             26
2      356317             8    4.250000                0                6                2  ...                 8                0                0                8          415             25
3      377644           181    4.685083                5              105               59  ...               145                1                6              174        38135             30
4      381533            12    4.75